In [ ]:
import numpy as np
import os
from datasets import load_dataset
import subprocess
from pandas import DataFrame
from pathlib import Path
import json
import textwrap
import coverage

In [7]:
ds = load_dataset("dz1/CodeScore-MBPP-ET")
CodeScore_df = DataFrame(ds['train'][89:99])
CodeScore_df

,text,code,task_id,test_setup_code,test_list,challenge_test_list,entry_point
0,Write a python function to find the length of ...,def len_log(list1):\r\n max=len(list1[0])\r...,90,,"[assert len_log([""python"",""PHP"",""bigdata""]) ==...",[],len_log
1,Write a function to check if a substring is pr...,"def find_substring(str1, sub_str):\r\n if an...",91,,"[assert find_substring([""red"", ""black"", ""white...",[],find_substring
2,Write a function to check whether the given nu...,def is_undulating(n): \r\n\tif (len(n) <= 2): ...,92,,"[assert is_undulating(""1212121"") == True, asse...",[],is_undulating
3,Write a function to calculate the value of 'a'...,"def power(a,b):\r\n\tif b==0:\r\n\t\treturn 1\...",93,,"[assert power(3,4) == 81, assert power(2,3) ==...",[],power
4,Write a function to extract the index minimum ...,from operator import itemgetter \r\ndef index_...,94,,"[assert index_minimum([('Rash', 143), ('Manjee...",[],index_minimum
5,Write a python function to find the minimum le...,def Find_Min_Length(lst): \r\n minLength =...,95,,"[assert Find_Min_Length([[1],[1,2]]) == 1, ass...",[],Find_Min_Length
6,Write a python function to find the number of ...,def divisor(n):\r\n for i in range(n):\r\n ...,96,,"[assert divisor(15) == 4 , assert divisor(12) ...",[],divisor
7,Write a function to find frequency count of li...,def frequency_lists(list1):\r\n list1 = [it...,97,,"[assert frequency_lists([[1, 2, 3, 2], [4, 5, ...",[],frequency_lists
8,Write a function to multiply all the numbers i...,def multiply_num(numbers): \r\n total = 1\...,98,,"[assert multiply_num((8, 2, 3, -1, 7))==-67.2,...",[],multiply_num
9,Write a function to convert the given decimal ...,def decimal_to_binary(n): \r\n return bin(n...,99,,"[assert decimal_to_binary(8) == '1000', assert...",[],decimal_to_binary


In [8]:
import google.generativeai as genai

def gemini_generate(prompt_technique,problemrange):
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel("gemini-2.5-flash")

    for test_idx in range(problemrange):
        response = model.generate_content(CodeScore_df["text_"+prompt_technique][test_idx])
        print(response.text)

        # Step 1: Remove Markdown fences if present
        raw_response = response.text
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            cleaned = "\n".join(cleaned.split("\n")[1:-1])  # remove first and last lines

        # Step 2: Parse JSON
        data = json.loads(cleaned)

        # Step 3: Extract the 'code' field
        code_text = data["code"]

        # Step 4: Write to file
        folder_path = Path("gemini_written_code/"+prompt_technique)
        folder_path.mkdir(parents=True, exist_ok=True)

        file_path = folder_path / (CodeScore_df['entry_point'][test_idx] + ".py")
        file_path.write_text(code_text, encoding="utf-8")

        print(f"Successfully wrote code to '{file_path}'")

In [98]:
def run_tests_visible(model_name,prompt_technique,problem_idx):
    entry_point = CodeScore_df['entry_point'][problem_idx]
    problem_test_list = CodeScore_df['test_list'][problem_idx]

    # Read the Python code from the file
    with open(model_name+"_written_code/"+prompt_technique+"/"+entry_point+".py", "r", encoding="utf-8") as f:
        problem_code = f.read()

    # Prepare namespace and execute the code
    namespace = {}
    exec(problem_code, namespace)

    for test in problem_test_list:
        try:
            exec(test, namespace)
            print(f"✅ Passed: {test}")
        except Exception as e:
            print(f"❌ Failed: {test}")
            print("   Error:", e)

# TEST EVERY TEST

def get_tests_passed(model_name,prompt_technique):
    folder_name= model_name+"_written_code/"+prompt_technique
    folder_path = Path(folder_name)
    problems_passed = []
    
    for idx, problem_name in enumerate(CodeScore_df['entry_point']):
        # Build the file path
        file_name = f"{problem_name}.py"
        file_path = folder_path / file_name

        if not file_path.exists():
            print(f"File not found: {file_path}")
            continue

        # Read the Python code from the file
        with open(file_path, "r", encoding="utf-8") as f:
            problem_code = f.read()

        # Get the corresponding tests from the dataframe
        problem_test_list = CodeScore_df['test_list'][idx]

        # Prepare namespace and execute the code
        namespace = {}
        try:
            exec(problem_code, namespace)
        except Exception as e:
            print(f"Error executing code for {problem_name} in {folder_name}: {e}")
            continue

        # Run tests
        passed = 0
        total = len(problem_test_list)

        for test in problem_test_list:
            try:
                exec(test, namespace)
                passed += 1
            except Exception:
                pass  # test failed
        
        if passed >= total: #passed all tests
            problems_passed.append(problem_name)

        print(f"{folder_name}/{file_name}: Passed {passed}/{total} tests")

    return problems_passed


In [11]:
# selfrepair_prompt_altered = """
# You are an expert Python programmer.
# Solve the following problem.

# Problem:
# {problem_statement}

# The name of the entry_point function should be "{entry_point}"

# If there are mistakes, repair them.

# Output solution in this exact JSON format:

# {{
#   "initial_code": "Write valid Python code implementing the plan.",
#   "repair_text": "Describe any mistakes in the initial_code, and explain how you'll repair them in the final code"
#   "code": "Write the final valid Python code implementing the plan.",
# }}

# Important:
# - The 'code' must be valid Python.
# - Do NOT include markdown fences (no ```python or ```).
# - Keep the JSON strictly valid.
# """

# scot_prompt = """
# You are an expert Python programmer.
# Solve the following problem, reasoning step-by-step in a structured way.

# Problem:
# {problem_statement}

# The name of the entry_point function should be "{entry_point}"

# Output your reasoning and solution in this exact JSON format:

# {{
#   "understanding": "Explain the problem in your own words.",
#   "plan": "Describe how you'll solve it.",
#   "reasoning_steps": ["Step 1...", "Step 2...", "..."],
#   "code": "Write valid Python code implementing the plan.",
# }}

# Important:
# - The 'code' must be valid Python.
# - Do NOT include markdown fences (no ```python or ```).
# - Keep the JSON strictly valid.
# """
# CodeScore_df['text_selfrepair'] = CodeScore_df.apply(lambda row: selfrepair_prompt.format(problem_statement=row.text, entry_point=row.entry_point), axis = 1)
# CodeScore_df['text_scot'] = CodeScore_df.apply(lambda row: scot_prompt.format(problem_statement=row.text,entry_point=row.entry_point), axis = 1)


# ASSIGNMENT 2 #

## Part 1 — Baseline Coverage (30% – 6 points)

1. Set up automated coverage collection for your A1 solutions (Use the tests provided with
your benchmark):
2. For each problem, report at least:
 - Number of tests passed
 - Line coverage and branch coverage (if the tool supports it for your language).
 - A one-line interpretation (e.g., “low branch coverage due to untested error path”).
3. Include a single summary table across all problems (problem → line %, branch %, notes)

In [144]:
# def test_suite_single_problem(model_name,prompt_technique,problem_idx, test_list):
#     folder_name= model_name+"_written_code/"+prompt_technique
#     folder_path = Path(folder_name)
#     problem_name = CodeScore_df['entry_point'][problem_idx]

#     file_name = f"{problem_name}.py"
#     file_path = folder_path / file_name

#     if not file_path.exists():
#         print(f"File not found: {file_path}")
#         return

#     with open(file_path, "r", encoding="utf-8") as f:
#         problem_code = f.read()

#     namespace = {}
#     try:
#         exec(problem_code, namespace)
#     except Exception as e:
#         print(f"Error executing code for {problem_name} in {folder_name}: {e}")
#         return

#     passed = 0
#     total = len(test_list)

#     for test in test_list:
#         try:
#             exec(test, namespace)
#             passed += 1
#         except Exception:
#             pass  # test failed

#     line_percent = -1
#     branch_percent = -1
#     test_percent = 100.0*(passed)/total

#     return {"problem_name": problem_name, "line_percent": line_percent, "branch_percent":branch_percent, "test_percent": test_percent }

def test_suite_single_problem(model_name, prompt_technique, problem_idx, test_list, CodeScore_df):
    folder_name = f"{model_name}_written_code/{prompt_technique}"
    folder_path = Path(folder_name)
    problem_name = CodeScore_df['entry_point'][problem_idx]

    file_name = f"{problem_name}.py"
    file_path = folder_path / file_name

    if not file_path.exists():
        print(f"File not found: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as f:
        problem_code = f.read()

    # --- Start coverage with branch tracking ---
    cov = coverage.Coverage(branch=True)
    cov.start()

    namespace = {}

    try:
        # Compile with filename so coverage recognizes it
        compiled_code = compile(problem_code, str(file_path), "exec")
        exec(compiled_code, namespace)
    except Exception as e:
        print(f"Error executing code for {problem_name} in {folder_name}: {e}")
        cov.stop()
        return

    passed = 0
    total = len(test_list)

    # Run tests
    for test in test_list:
        try:
            exec(test, namespace)
            passed += 1
        except Exception:
            pass

    # Stop coverage collection
    cov.stop()
    cov.save()

    # --- Export coverage data as JSON ---
    json_path = Path("tmp_coverage.json")
    cov.json_report(outfile=str(json_path))

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Get per-file data
    file_data = data["files"].get(str(file_path))
    if not file_data:
        return {
            "problem": problem_name,
            "#passed": 0,
            "line%": 0.0,
            "branch%": 0.0,
            "test%": 0.0,
            "metric": abs(test_percent-branch_percent-line_percent)*test_percent/100
        }

    summary = file_data["summary"]

    # --- Extract metrics cleanly ---
    line_percent = summary.get("percent_covered", 0.0)
    covered_branches = summary.get("covered_branches", 0)
    total_branches = summary.get("num_branches", 0)
    branch_percent = (100.0 * covered_branches / total_branches) if total_branches else 0.0
    test_percent = 100.0 * passed / total if total > 0 else 0.0

    return {
        "problem": problem_name,
        "#passed": passed,
        "line%": line_percent,
        "branch%": branch_percent,
        "test%": test_percent,
        "metric": abs(test_percent-branch_percent-line_percent)*test_percent/100
    }
  
def test_suite(model_name, prompt_technique, CodeScore_df):
    folder_name= model_name+"_written_code/"+prompt_technique
    test_results = []
    
    for idx in range(len(CodeScore_df['entry_point'])):
        # Get the corresponding tests from the dataframe
        problem_test_list = CodeScore_df['test_list'][idx]

        test_result = test_suite_single_problem(model_name,prompt_technique,idx,problem_test_list, CodeScore_df)

        test_results.append(test_result)
        
    return test_results

def run_tests_visible(model_name,prompt_technique,problem_idx, test_list):
    entry_point = CodeScore_df['entry_point'][problem_idx]
    problem_test_list = test_list

    # Read the Python code from the file
    with open(model_name+"_written_code/"+prompt_technique+"/"+entry_point+".py", "r", encoding="utf-8") as f:
        problem_code = f.read()

    # Prepare namespace and execute the code
    namespace = {}
    exec(problem_code, namespace)

    for test in problem_test_list:
        try:
            exec(test, namespace)
            print(f"✅ Passed: {test}")
        except Exception as e:
            print(f"❌ Failed: {test}")
            print("   Error:", e)

coverage metric = abs(test_percent-branch_percent-line_percent)*test_percent/100

In [134]:
print(test_suite_single_problem("gemini_run_1","scot",2, CodeScore_df['test_list'][2],CodeScore_df))
print(test_suite_single_problem("gemini_run_1","selfrepair",9,CodeScore_df['test_list'][9],CodeScore_df))

{'problem': 'is_undulating', '#passed': 85, 'line%': 93.10344827586206, 'branch%': 90.0, 'test%': 83.33333333333333, 'metric': 83.14176245210727}
{'problem': 'decimal_to_binary', '#passed': 3, 'line%': 84.61538461538461, 'branch%': 75.0, 'test%': 2.9411764705882355, 'metric': 4.608064945435188}


In [135]:
print("\ngemini selfrepair\n")
res = test_suite("gemini_run_1","selfrepair", CodeScore_df)
print(DataFrame(res))

print("\ngemini selfrepair_with_test\n")
res = test_suite("gemini","selfrepair_with_test", CodeScore_df)
print(DataFrame(res))

print("\ngemini scot\n")
res = test_suite("gemini_run_1","scot", CodeScore_df)
print(DataFrame(res))

print("\ngemini scot_with_test\n")
res = test_suite("gemini","scot_with_test", CodeScore_df)
print(DataFrame(res))


gemini selfrepair

             problem  #passed       line%     branch%       test%      metric
0            len_log        0   28.571429    0.000000    0.000000    0.000000
1     find_substring        0  100.000000    0.000000    0.000000    0.000000
2      is_undulating      102   92.307692   91.666667  100.000000   83.974359
3              power      102  100.000000    0.000000  100.000000    0.000000
4      index_minimum        0   66.666667   50.000000    0.000000    0.000000
5    Find_Min_Length      102  100.000000  100.000000  100.000000  100.000000
6            divisor      102   83.333333   80.000000  100.000000   63.333333
7    frequency_lists        0  100.000000  100.000000    0.000000    0.000000
8       multiply_num      102   84.615385   75.000000  100.000000   59.615385
9  decimal_to_binary        3   84.615385   75.000000    2.941176    4.608065

gemini selfrepair_with_test

             problem  #passed       line%     branch%       test%      metric
0            l

In [136]:
print("\nclaude selfrepair\n")
res = test_suite("claude_run_1","selfrepair", CodeScore_df)
print(DataFrame(res))

print("\nclaude selfrepair_with_test\n")
res = test_suite("claude","selfrepair_with_test", CodeScore_df)
print(DataFrame(res))

print("\nclaude scot\n")
res = test_suite("claude_run_1","scot", CodeScore_df)
print(DataFrame(res))

print("\nclaude scot_with_test\n")
res = test_suite("claude","scot_with_test", CodeScore_df)
print(DataFrame(res))


claude selfrepair

             problem  #passed       line%  branch%       test%      metric
0            len_log      102   60.000000     50.0  100.000000   10.000000
1     find_substring      102  100.000000    100.0  100.000000  100.000000
2      is_undulating      102   88.888889     87.5  100.000000   76.388889
3              power      102  100.000000      0.0  100.000000    0.000000
4      index_minimum        0   66.666667     50.0    0.000000    0.000000
5    Find_Min_Length      102   66.666667     50.0  100.000000   16.666667
6            divisor      102   90.000000     87.5  100.000000   77.500000
7    frequency_lists        0  100.000000      0.0    0.000000    0.000000
8       multiply_num      102   81.818182     75.0  100.000000   56.818182
9  decimal_to_binary        3   66.666667     50.0    2.941176    3.344867

claude selfrepair_with_test

             problem  #passed       line%     branch%       test%      metric
0            len_log      102  100.000000  100.

For the following parts you will only work with 2 set of (problem statement, benchmark test
suite, LLM generated solution). Choose ones with room for improvement. Here’s an
example: Chose highest |%test - %branch-coverage| x %test (You can use your own metric to
choose the two problems)


Problem 1: 2 set of (problem statement, benchmark test suite, LLM generated solution)


decimal_to_binary - default test suite -  selfrepair (gemini)


is_undulating - default test suite -  scot (gemini)

## Part 2 — LLM-Assisted Test Generation & Coverage Improvement (50% – 10 points) ##

In [36]:
test_generation_prompt_template = """
You are an expert Python programmer.
Create tests for the following problem.

Problem:
{problem_statement}

The name of the function to test is "{entry_point}"

Here are previous tests to understand the format of the function: "{test_list_sample}"
You must add additional tests not matching these tests to increase coverage.

Output solution in this exact JSON format:

{{
  "test_list": Array of tests using assert format.,
}}

Important:
- The 'code' must be valid Python.
- Do NOT include markdown fences (no ```python or ```).
- Keep the JSON strictly valid.
"""

In [ ]:
def gemini_generate_automated_tests(prompt_template, problem_statement, entry_point, test_list_sample):
    import google.generativeai as genai

    formatted_prompt = prompt_template.format(problem_statement=problem_statement, entry_point=entry_point, test_list_sample=test_list_sample)

    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel("gemini-2.5-flash")

    response = model.generate_content(formatted_prompt)
    print(response.text)

    # Remove Markdown fences if present
    raw_response = response.text
    cleaned = raw_response.strip()
    if cleaned.startswith("```"):
        cleaned = "\n".join(cleaned.split("\n")[1:-1])  # remove first and last lines

    data = json.loads(cleaned)

    code_text = data["test_list"]

    print(f"Successfully wrote tests for '{entry_point}'")

    return code_text

In [38]:
undulate_text = CodeScore_df["text"][2]
undulate_entry_point = CodeScore_df["entry_point"][2]
undulate_test_sample = CodeScore_df["test_list"][2][0]

bin_text = CodeScore_df["text"][9]
bin_entry_point = CodeScore_df["entry_point"][9]
bin_test_sample = CodeScore_df["test_list"][9][0]

In [39]:
bin_generated_tests = gemini_generate_automated_tests(test_generation_prompt_template, bin_text, bin_entry_point, bin_test_sample)
undulating_generated_tests = gemini_generate_automated_tests(test_generation_prompt_template, undulate_text, undulate_entry_point, undulate_test_sample)

{
  "test_list": [
    "assert decimal_to_binary(0) == '0'",
    "assert decimal_to_binary(1) == '1'",
    "assert decimal_to_binary(2) == '10'",
    "assert decimal_to_binary(3) == '11'",
    "assert decimal_to_binary(4) == '100'",
    "assert decimal_to_binary(5) == '101'",
    "assert decimal_to_binary(7) == '111'",
    "assert decimal_to_binary(10) == '1010'",
    "assert decimal_to_binary(16) == '10000'",
    "assert decimal_to_binary(25) == '11001'",
    "assert decimal_to_binary(123) == '1111011'",
    "assert decimal_to_binary(255) == '11111111'",
    "assert decimal_to_binary(256) == '100000000'"
  ]
}
Successfully wrote tests for 'decimal_to_binary'
```json
{
  "test_list": [
    "assert is_undulating(\"1212121\") == True",
    "assert is_undulating(\"4545\") == True",
    "assert is_undulating(\"78787\") == True",
    "assert is_undulating(\"99999\") == True",
    "assert is_undulating(\"2222\") == True",
    "assert is_undulating(\"101\") == True",
    "assert is_undulating

Convergence criteria: 3 consecutive iterations should have increase in %coverage less than 3%. Aka Coverage(i) - Coverage(i-2) <= 3%

In [138]:
print(test_suite_single_problem("gemini_run_1","selfrepair",2, undulating_generated_tests, CodeScore_df))
print(test_suite_single_problem("gemini_run_1","scot",9,bin_generated_tests,CodeScore_df))

{'problem': 'is_undulating', '#passed': 15, 'line%': 100.0, 'branch%': 100.0, 'test%': 83.33333333333333, 'metric': 97.22222222222223}
{'problem': 'decimal_to_binary', '#passed': 13, 'line%': 100.0, 'branch%': 100.0, 'test%': 100.0, 'metric': 100.0}


In [67]:
def gemini_run_until_test_convergence(prompt_template, problem_statement, entry_point, test_list_sample, model_name, prompt_technique,problem_idx): 
    total_test_list = [test_list_sample] # start with a single sample.
    # 3 consecutive iterations should have increase in %coverage less than 3%
    iteration_coverage_queue = [-100,-100,-100]

    total_coverage_difference_percentage = 100 # ideal: .03
    iterations = 0

    while(total_coverage_difference_percentage > 3):
        iterations += 1
        generated_tests = gemini_generate_automated_tests(prompt_template, problem_statement, entry_point, total_test_list)

        # add generated tests to total list
        total_test_list = total_test_list + generated_tests
        
        # DEDUPE PROCESS
        dedupe_total_test_list = set(total_test_list) # remove duplicates
        dedupe_total_test_list= list(dedupe_total_test_list)
        print("removed "  + str(len(total_test_list) - len(dedupe_total_test_list)) + " duplicate tests")
        total_test_list = dedupe_total_test_list

        # TEST
        cur_test_result = test_suite_single_problem(model_name,prompt_technique,problem_idx,total_test_list)
        
        # REPORT COVERAGE
        cur_line_percent = cur_test_result["line_percent"]
        cur_branch_percent = cur_test_result["branch_percent"]
        cur_test_percent = cur_test_result["test_percent"]

        coverage = cur_test_percent # assigning coverage as tests passed.

        iteration_coverage_queue.append(coverage)
        removed = iteration_coverage_queue.pop(0)

        total_coverage_difference_percentage = abs(iteration_coverage_queue[2] - iteration_coverage_queue[0])
        
        print("current number of tests: " + str(len(total_test_list)))
        print(cur_test_result)
        print("last two coverages: " + str(iteration_coverage_queue))
        print("total coverage percentage difference : |" + str(iteration_coverage_queue[2]) + "% - " + str(iteration_coverage_queue[0]) + "%| = " + str(total_coverage_difference_percentage) + "%")

    return total_test_list

In [68]:
bin_tests = gemini_run_until_test_convergence(test_generation_prompt_template, bin_text, bin_entry_point, bin_test_sample, "gemini_run_1","selfrepair",9)

{
  "test_list": [
    "assert decimal_to_binary(0) == '0'",
    "assert decimal_to_binary(1) == '1'",
    "assert decimal_to_binary(2) == '10'",
    "assert decimal_to_binary(3) == '11'",
    "assert decimal_to_binary(4) == '100'",
    "assert decimal_to_binary(5) == '101'",
    "assert decimal_to_binary(7) == '111'",
    "assert decimal_to_binary(10) == '1010'",
    "assert decimal_to_binary(15) == '1111'",
    "assert decimal_to_binary(16) == '10000'",
    "assert decimal_to_binary(32) == '100000'",
    "assert decimal_to_binary(64) == '1000000'",
    "assert decimal_to_binary(100) == '1100100'",
    "assert decimal_to_binary(255) == '11111111'",
    "assert decimal_to_binary(1024) == '10000000000'"
  ]
}
Successfully wrote tests for 'decimal_to_binary'
removed 0 duplicate tests
current number of tests: 16
{'problem_name': 'decimal_to_binary', 'line_percent': -1, 'branch_percent': -1, 'test_percent': 100.0}
last two coverages: [-100, -100, 100.0]
total coverage percentage difference

In [ ]:
bin_tests

["assert decimal_to_binary(1) == '1'",
 "assert decimal_to_binary(100) == '1100100'",
 "assert decimal_to_binary(42) == '101010'",
 "assert decimal_to_binary(256) == '100000000'",
 "assert decimal_to_binary(2048) == '100000000000'",
 "assert decimal_to_binary(3) == '11'",
 "assert decimal_to_binary(12) == '1100'",
 "assert decimal_to_binary(18) == '10010'",
 "assert decimal_to_binary(30) == '11110'",
 "assert decimal_to_binary(64) == '1000000'",
 "assert decimal_to_binary(123) == '1111011'",
 "assert decimal_to_binary(20) == '10100'",
 "assert decimal_to_binary(19) == '10011'",
 "assert decimal_to_binary(8) == '1000'",
 "assert decimal_to_binary(17) == '10001'",
 "assert decimal_to_binary(400) == '110010000'",
 "assert decimal_to_binary(0) == '0'",
 "assert decimal_to_binary(255) == '11111111'",
 "assert decimal_to_binary(99) == '1100011'",
 "assert decimal_to_binary(1025) == '10000000001'",
 "assert decimal_to_binary(73) == '1001001'",
 "assert decimal_to_binary(14) == '1110'",
 "asse

In [146]:
len(bin_tests)
run_tests_visible("gemini_run_1","selfrepair",9,bin_tests)

✅ Passed: assert decimal_to_binary(1) == '1'
✅ Passed: assert decimal_to_binary(100) == '1100100'
✅ Passed: assert decimal_to_binary(42) == '101010'
✅ Passed: assert decimal_to_binary(256) == '100000000'
✅ Passed: assert decimal_to_binary(2048) == '100000000000'
✅ Passed: assert decimal_to_binary(3) == '11'
✅ Passed: assert decimal_to_binary(12) == '1100'
✅ Passed: assert decimal_to_binary(18) == '10010'
✅ Passed: assert decimal_to_binary(30) == '11110'
✅ Passed: assert decimal_to_binary(64) == '1000000'
✅ Passed: assert decimal_to_binary(123) == '1111011'
✅ Passed: assert decimal_to_binary(20) == '10100'
✅ Passed: assert decimal_to_binary(19) == '10011'
✅ Passed: assert decimal_to_binary(8) == '1000'
✅ Passed: assert decimal_to_binary(17) == '10001'
✅ Passed: assert decimal_to_binary(400) == '110010000'
✅ Passed: assert decimal_to_binary(0) == '0'
✅ Passed: assert decimal_to_binary(255) == '11111111'
✅ Passed: assert decimal_to_binary(99) == '1100011'
✅ Passed: assert decimal_to_binar

In [70]:
undulate_tests = gemini_run_until_test_convergence(test_generation_prompt_template, undulate_text, undulate_entry_point, undulate_test_sample, "gemini_run_1","scot",2)

```json
{
  "test_list": [
    "assert is_undulating(\"12\") == True",
    "assert is_undulating(\"343\") == True",
    "assert is_undulating(\"9090\") == True",
    "assert is_undulating(\"5656565\") == True",
    "assert is_undulating(\"0101\") == True",
    "assert is_undulating(\"1\") == False",
    "assert is_undulating(\"11\") == False",
    "assert is_undulating(\"122\") == False",
    "assert is_undulating(\"123\") == False",
    "assert is_undulating(\"1213\") == False",
    "assert is_undulating(\"77777\") == False",
    "assert is_undulating(\"12123\") == False",
    "assert is_undulating(\"45454545\") == True"
  ]
}
```
Successfully wrote tests for 'is_undulating'
removed 0 duplicate tests
current number of tests: 14
{'problem_name': 'is_undulating', 'line_percent': -1, 'branch_percent': -1, 'test_percent': 85.71428571428571}
last two coverages: [-100, -100, 85.71428571428571]
total coverage percentage difference : |85.71428571428571% - -100%| = 185.71428571428572%
```json


In [145]:
len(undulate_tests)
run_tests_visible("gemini_run_1","scot",2,undulate_tests)

✅ Passed: assert is_undulating("87878787") == True
✅ Passed: assert is_undulating("123123") == False
✅ Passed: assert is_undulating("45454545") == True
❌ Failed: assert is_undulating("1213") == False
   Error: 
✅ Passed: assert is_undulating("1221") == False
✅ Passed: assert is_undulating("1122") == False
❌ Failed: assert is_undulating("121213") == False
   Error: 
✅ Passed: assert is_undulating("2222") == False
✅ Passed: assert is_undulating("9090") == True
✅ Passed: assert is_undulating("1010101010101010101") == True
✅ Passed: assert is_undulating("122") == False
✅ Passed: assert is_undulating("334") == False
✅ Passed: assert is_undulating("001") == False
✅ Passed: assert is_undulating("78787") == True
✅ Passed: assert is_undulating("12312") == False
✅ Passed: assert is_undulating("5656565") == True
✅ Passed: assert is_undulating("989898989") == True
✅ Passed: assert is_undulating("10100") == False
✅ Passed: assert is_undulating("12345") == False
✅ Passed: assert is_undulating("") ==

In [142]:
print(test_suite_single_problem("gemini_run_1","scot",2, undulate_tests, CodeScore_df))
print(test_suite_single_problem("gemini_run_1","selfrepair",9,bin_tests, CodeScore_df))

{'problem': 'is_undulating', '#passed': 31, 'line%': 100.0, 'branch%': 100.0, 'test%': 88.57142857142857, 'metric': 98.69387755102042}
{'problem': 'decimal_to_binary', '#passed': 51, 'line%': 100.0, 'branch%': 100.0, 'test%': 100.0, 'metric': 100.0}


In [143]:
default_test_list_bin = CodeScore_df['test_list'][9]
default_test_list_undulate = CodeScore_df['test_list'][2]

print(test_suite_single_problem("gemini_run_1","scot",2, default_test_list_undulate, CodeScore_df))
print(test_suite_single_problem("gemini_run_1","selfrepair",9,default_test_list_bin, CodeScore_df))

{'problem': 'is_undulating', '#passed': 85, 'line%': 93.10344827586206, 'branch%': 90.0, 'test%': 83.33333333333333, 'metric': 83.14176245210727}
{'problem': 'decimal_to_binary', '#passed': 3, 'line%': 84.61538461538461, 'branch%': 75.0, 'test%': 2.9411764705882355, 'metric': 4.608064945435188}


## Part 3 — Fault Detection Check (20% – 4 points) ##

In [148]:
bin_code = """
def decimal_to_binary(n):
    if n == 0:
        return "0"
    
    binary_digits = []
    while n > 0:
        remainder = n % 2
        binary_digits.append(str(remainder))
        n = n // 2
    
    # Repair: Reverse the list of digits before joining them
    return "".join(binary_digits[::-1])
"""
undulate_code = """
def is_undulating(num):
    # Convert the number to a list of integer digits
    s_num = str(num)
    digits = [int(d) for d in s_num]
    n = len(digits)

    # An undulating number requires at least 3 digits to establish the alternating pattern
    # Also handles num=0, which has 1 digit, correctly returning False.
    if n < 3:
        return False

    # Check the relation between the first two digits
    # If they are equal, it cannot be undulating (strict inequality required)
    if digits[0] == digits[1]:
        return False

    # prev_was_less tracks the relation for the previous pair of digits
    # True if digits[k-1] < digits[k], False if digits[k-1] > digits[k]
    prev_was_less = (digits[0] < digits[1])

    # Iterate from the second digit to the second-to-last digit
    # We compare digits[i] and digits[i+1]
    for i in range(1, n - 1):
        d_curr = digits[i]
        d_next = digits[i+1]

        # If adjacent digits are equal, it's not undulating
        if d_curr == d_next:
            return False
        
        # Determine the current relation
        current_is_less = (d_curr < d_next)

        # If the current relation is the same as the previous one,
        # the alternating pattern is broken (e.g., < < or > >)
        if current_is_less == prev_was_less:
            return False
        
        # Update prev_was_less for the next iteration (relation should flip)
        prev_was_less = current_is_less
    
    # If the loop completes, the number is undulating
    return True
"""

In [147]:
new_bin_code = """
def decimal_to_binary(n):
    # no case for n = 0 
    
    binary_digits = []
    while n >= 0:
        remainder = n % 2
        binary_digits.append(str(remainder))
        n = n // 2
    
    # Repair: Reverse the list of digits before joining them
    return "".join(binary_digits[::-1])
"""
new_undulate_code = """
def is_undulating(num):
    # Convert the number to a list of integer digits
    s_num = str(num)
    digits = [int(d) for d in s_num]
    n = len(digits)

    # An undulating number requires at least 3 digits to establish the alternating pattern
    # Also handles num=0, which has 1 digit, correctly returning False.
    if n < 3:
        return False

    # Check the relation between the first two digits
    # If they are equal, it cannot be undulating (strict inequality required)
    if digits[0] == digits[1]:
        return False

    # prev_was_less tracks the relation for the previous pair of digits
    # True if digits[k-1] < digits[k], False if digits[k-1] > digits[k]
    prev_was_less = (digits[0] < digits[1])

    # Iterate from the second digit to the second-to-last digit
    # We compare digits[i] and digits[i+1]
    for i in range(1, n - 1):
        d_curr = digits[i]
        d_next = digits[i+1]

        # If adjacent digits are equal, it's not undulating
        if d_curr == d_next:
            return False
        
        # Determine the current relation
        current_is_less = (d_curr < d_next)

        # If the current relation is the same as the previous one,
        # the alternating pattern is broken (e.g., < < or > >)
        if current_is_less == prev_was_less:
            return False
        
        # DONT Update prev_was_less for the next iteration (relation should flip)
        # prev_was_less = current_is_less
    
    # If the loop completes, the number is undulating
    return True
"""

In [152]:
default_test_list_bin = CodeScore_df['test_list'][9]
default_test_list_undulate = CodeScore_df['test_list'][2]

print(test_suite_single_problem("gemini_run_1","scot",2, default_test_list_undulate,CodeScore_df))
print(test_suite_single_problem("gemini_run_1","selfrepair",9,default_test_list_bin,CodeScore_df))
print(test_suite_single_problem("gemini_run_1","scot",2, undulate_tests,CodeScore_df))
print(test_suite_single_problem("gemini_run_1","selfrepair",9,bin_tests,CodeScore_df))

{'problem': 'is_undulating', '#passed': 92, 'line%': 92.85714285714286, 'branch%': 90.0, 'test%': 90.19607843137256, 'metric': 83.57664634481243}
{'problem': 'decimal_to_binary', '#passed': 3, 'line%': 100.0, 'branch%': 100.0, 'test%': 2.9411764705882355, 'metric': 5.795847750865053}
{'problem': 'is_undulating', '#passed': 23, 'line%': 100.0, 'branch%': 100.0, 'test%': 65.71428571428571, 'metric': 88.24489795918366}
{'problem': 'decimal_to_binary', '#passed': 50, 'line%': 100.0, 'branch%': 100.0, 'test%': 98.03921568627452, 'metric': 99.96155324875048}


In [150]:
run_tests_visible("gemini_run_1","selfrepair",9,bin_tests)

✅ Passed: assert decimal_to_binary(1) == '1'
✅ Passed: assert decimal_to_binary(100) == '1100100'
✅ Passed: assert decimal_to_binary(42) == '101010'
✅ Passed: assert decimal_to_binary(256) == '100000000'
✅ Passed: assert decimal_to_binary(2048) == '100000000000'
✅ Passed: assert decimal_to_binary(3) == '11'
✅ Passed: assert decimal_to_binary(12) == '1100'
✅ Passed: assert decimal_to_binary(18) == '10010'
✅ Passed: assert decimal_to_binary(30) == '11110'
✅ Passed: assert decimal_to_binary(64) == '1000000'
✅ Passed: assert decimal_to_binary(123) == '1111011'
✅ Passed: assert decimal_to_binary(20) == '10100'
✅ Passed: assert decimal_to_binary(19) == '10011'
✅ Passed: assert decimal_to_binary(8) == '1000'
✅ Passed: assert decimal_to_binary(17) == '10001'
✅ Passed: assert decimal_to_binary(400) == '110010000'
❌ Failed: assert decimal_to_binary(0) == '0'
   Error: 
✅ Passed: assert decimal_to_binary(255) == '11111111'
✅ Passed: assert decimal_to_binary(99) == '1100011'
✅ Passed: assert decim

In [151]:
run_tests_visible("gemini_run_1","scot",2,undulate_tests)

❌ Failed: assert is_undulating("87878787") == True
   Error: 
✅ Passed: assert is_undulating("123123") == False
❌ Failed: assert is_undulating("45454545") == True
   Error: 
✅ Passed: assert is_undulating("1213") == False
✅ Passed: assert is_undulating("1221") == False
✅ Passed: assert is_undulating("1122") == False
✅ Passed: assert is_undulating("121213") == False
✅ Passed: assert is_undulating("2222") == False
❌ Failed: assert is_undulating("9090") == True
   Error: 
❌ Failed: assert is_undulating("1010101010101010101") == True
   Error: 
✅ Passed: assert is_undulating("122") == False
✅ Passed: assert is_undulating("334") == False
✅ Passed: assert is_undulating("001") == False
❌ Failed: assert is_undulating("78787") == True
   Error: 
✅ Passed: assert is_undulating("12312") == False
❌ Failed: assert is_undulating("5656565") == True
   Error: 
❌ Failed: assert is_undulating("989898989") == True
   Error: 
✅ Passed: assert is_undulating("10100") == False
✅ Passed: assert is_undulating(